### Chapter 3.3 - Synthetic Regression Data

Synthetic data is data we create ourselves. It lets us test whether a learning algorithm can recover a known pattern before trusting it on messy real data.


In [ ]:
import math
import random
import numpy as np
import torch


### 3.3.1 Generating the Dataset

#### 1. Intuition

Synthetic regression data is generated from a known formula plus noise.

A true parameter is the parameter used to create the fake labels. Noise is random variation added so the task is not perfectly clean.

#### 2. Why this exists

If we know the true weights and bias, we can check whether training recovers them. This is a controlled test of the learning pipeline.


#### 3. Examples

Manual Python: create three labels from one feature.


In [ ]:
x_values = [1.0, 2.0, 3.0]
true_w = 2.0
true_b = 1.0
labels = []
for x in x_values:
    labels.append(true_w * x + true_b)
labels


PyTorch: generate a small noisy linear dataset.


In [ ]:
torch.manual_seed(0)
X = torch.randn(5, 2)
true_w = torch.tensor([2.0, -3.4])
true_b = 4.2
noise = torch.randn(5) * 0.01
y = X @ true_w + true_b + noise
X, y


#### 4. Step-by-step breakdown

The manual example uses an exact line with no noise.

`torch.manual_seed(0)` makes the random numbers repeatable for debugging.

`torch.randn(5, 2)` creates 5 examples with 2 features each.

`X @ true_w + true_b` creates clean labels from the true linear rule.

`noise` adds tiny random errors so the data looks more realistic.

#### 5. Connection to ML systems

Synthetic data is a unit test for training code. If a model cannot learn a simple generated line, the bug is probably in the implementation rather than the dataset.

#### 6. Common confusion points

- Synthetic data is useful because the true answer is known.
- Noise makes exact recovery impossible, but close recovery should still happen.
- A random seed helps reproduce debugging results.
- Synthetic success does not guarantee real-world success.


### 3.3.2 Reading the Dataset

#### 1. Intuition

Reading a dataset means repeatedly selecting examples and labels for training.

A minibatch is a small group of examples used for one update. Shuffling means changing example order randomly.

#### 2. Why this exists

Training on all examples at once can be expensive. Training one example at a time can be noisy. Minibatches balance efficiency and randomness.


#### 3. Examples

Manual minibatch reader using shuffled indices.


In [ ]:
indices = list(range(len(y)))
random.shuffle(indices)
batch_size = 2
batch_idx = indices[:batch_size]
batch_X = X[batch_idx]
batch_y = y[batch_idx]
batch_X.shape, batch_y.shape


A tiny generator that yields batches.


In [ ]:
def data_iter(X, y, batch_size):
    indices = list(range(len(y)))
    random.shuffle(indices)
    for start in range(0, len(y), batch_size):
        idx = indices[start:start + batch_size]
        yield X[idx], y[idx]

next(data_iter(X, y, 2))


#### 4. Step-by-step breakdown

`range(len(y))` creates one index per label.

`random.shuffle(indices)` changes the order in place.

`batch_idx` selects the first few shuffled positions.

`X[batch_idx]` selects matching input rows.

`yield` makes a generator. A generator returns one item at a time when asked, instead of building all batches at once.

#### 5. Connection to ML systems

A DataLoader is the framework version of this idea. It repeatedly returns batches while handling shuffling and batching details.

#### 6. Common confusion points

- Shuffle indices, not features and labels separately.
- Batch size controls how many examples are in one update.
- The last batch may be smaller than the others.
- A generator pauses after `yield` and resumes when the next item is requested.


### 3.3.3 Concise Implementation of the Data Loader

#### 1. Intuition

A concise implementation uses framework data utilities instead of a handwritten batch generator.

In PyTorch, a `TensorDataset` stores aligned tensors. A `DataLoader` groups dataset items into batches.

#### 2. Why this exists

Framework data loaders reduce repeated boilerplate and handle common cases correctly, such as shuffling and batching.


#### 3. Examples

PyTorch DataLoader version.


In [ ]:
dataset = torch.utils.data.TensorDataset(X, y)
loader = torch.utils.data.DataLoader(dataset, batch_size=2, shuffle=True)
for batch_X, batch_y in loader:
    first_shapes = batch_X.shape, batch_y.shape
    break
first_shapes


#### 4. Step-by-step breakdown

`TensorDataset(X, y)` stores features and labels together by row position.

`DataLoader(..., batch_size=2, shuffle=True)` creates an object that can produce shuffled batches.

The `for` loop asks the loader for one batch at a time.

The `break` stops after the first batch so the example stays small.

#### 5. Connection to ML systems

Most PyTorch training code uses DataLoader objects. The important mapping is simple: DataLoader returns the same kind of `(batch_X, batch_y)` pairs as our manual generator.

#### 6. Common confusion points

- DataLoader is an iterator-like object, not the data itself.
- TensorDataset assumes tensors are aligned along the first dimension.
- Shuffling changes order, not values.
- Concise code should still be explainable using the manual version.


### 3.3.4 Summary

#### 1. Intuition

Synthetic data gives a controlled setting for debugging linear regression.

The workflow is: choose true parameters, generate features, generate labels, add noise, then read data in batches.

#### 2. Why this exists

A known data-generating process makes implementation mistakes easier to find.


#### 3. Examples

One compact synthetic-data function.


In [ ]:
def synthetic_data(w, b, n):
    X = torch.randn(n, len(w))
    y = X @ w + b
    y += torch.randn(n) * 0.01
    return X, y

X_small, y_small = synthetic_data(torch.tensor([2.0, -1.0]), 0.5, 4)
X_small.shape, y_small.shape


#### 4. Step-by-step breakdown

The function receives true weights, true bias, and number of examples.

It creates feature rows using random normal values.

It computes labels from the linear rule and adds noise.

It returns features and labels as tensors.

#### 5. Connection to ML systems

This function can feed later from-scratch and concise implementations, letting both versions learn from the same controlled data.

#### 6. Common confusion points

- A data generator is not a trained model.
- The true weights create labels; learned weights try to recover them.
- Noise prevents perfect zero loss.
- Always check returned shapes.


### 3.3.5 Exercises

#### 1. Intuition

The exercises test whether you can create and read a synthetic regression dataset deliberately.

#### 2. Why this exists

Synthetic datasets are most useful when you can change one part and predict the effect.


#### 3. Examples

Exercise 1: generate data with three input features.


In [ ]:
true_w = torch.tensor([1.0, -2.0, 0.5])
X3, y3 = synthetic_data(true_w, b=1.5, n=6)
X3.shape, y3.shape


Exercise 2: read one minibatch from the generated data.


In [ ]:
loader = torch.utils.data.DataLoader(
    torch.utils.data.TensorDataset(X3, y3), batch_size=3, shuffle=True)
batch_X, batch_y = next(iter(loader))
batch_X.shape, batch_y.shape


#### 4. Step-by-step breakdown

Exercise 1 checks that feature count controls the second dimension of `X`.

Exercise 2 checks DataLoader batching.

The first dimension is the number of examples, and the second dimension is the number of features.

#### 5. Connection to ML systems

These skills support all upcoming training notebooks, where data must arrive as correctly shaped minibatches.

#### 6. Common confusion points

- `n` controls rows, not columns.
- `len(w)` controls feature count.
- `iter(loader)` creates an iterator over batches.
- `next(...)` asks for one batch from that iterator.
